# pic2 phase 1 — Moment deposition (CIC)

This notebook tests the **right box** of Screenshot 1 (*Accumulate moments*):

$$N_j = \sum_p w_p\, W(\mathbf r_p - \mathbf r_j)\,/\,dV, \qquad \mathbf u_j = \sum_p w_p \mathbf v_p\, W(\mathbf r_p - \mathbf r_j)\,/\,N_j$$

$W$ is the cloud-in-cell tent: each particle touches its 8 surrounding nodes. The C++ lives in `src/moments.hpp` (`Moments::deposit(pop)` — one argument, since `Moments` owns its `GridGeom` and `Population` owns the weights `w`).

**Abstract API rule used throughout pic2:** geometry and populations travel as *objects*, so no function takes more than 3 args (`GridGeom::idx(i,j,k)` is the max). The only exception is `Population.from_arrays(x,v,q,m,w)`, where the five numpy arrays must enter together.

Tests T0–T5 check exactness, T6 the PPC vocabulary stub (`predictor1/predictor2/corrector`), T7 the statistical signature $\sigma_n/\bar n \propto 1/\sqrt{N}$.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("build"))
import numpy as np
import matplotlib.pyplot as plt
import pic2_ext as px

L = 2.0 * np.pi
N = 4  # cells per axis for the exact tests

def make_geom(n=N, L=L):
    """Cubic periodic domain [0, L)^3 with n^3 nodes — 2 args, geometry is one object."""
    g = px.Geom()
    g.origin = px.Vec3(0.0, 0.0, 0.0)
    d = L / n
    g.d = px.Vec3(d, d, d)
    g.nx = g.ny = g.nz = n
    assert g.ncells() == n**3
    return g

def make_pop(x, v, w):
    x = np.ascontiguousarray(x, dtype=float)
    v = np.ascontiguousarray(v, dtype=float)
    n = x.shape[0]
    return px.Population.from_arrays(x, v, np.ones(n), np.ones(n),
                                    np.ascontiguousarray(w, dtype=float))

def flat(i, j, k, n=N):
    return (i % n * n + j % n) * n + k % n

g = make_geom()
dV = (L / N) ** 3
w0 = 1.7
print("geom cells:", g.ncells(), "cell volume dV =", dV)

## T0 — Particle exactly on a node

All CIC weights are 1 on that node and 0 elsewhere, so the full weight $w_0$ lands on node $(0,0,0)$ as density $w_0/dV$, and the nodal velocity equals the particle velocity. Nothing may leak to other nodes.

In [ ]:
pop = make_pop([[0.0, 0.0, 0.0]], [[1.0, 2.0, 3.0]], [w0])
out = px.deposit_moments(pop, g)
assert np.isclose(out["n"][flat(0, 0, 0)], w0 / dV), out["n"].max()
assert np.isclose(out["n"].sum() * dV, w0)
assert np.allclose(out["u"][flat(0, 0, 0)], [1.0, 2.0, 3.0])
print("T0 exact-node ok: n[node] =", out["n"][flat(0, 0, 0)], "expected", w0 / dV)

## T1 — Particle at a cell center

At the cell center every 1D tent factor is $1/2$, so the product weight is $(1/2)^3 = 1/8$ on each of the 8 corners. Each touched node gets $w_0/8/dV$ and reports the particle velocity.

In [ ]:
c = L / N * 0.5
pop = make_pop([[c, c, c]], [[0.0, 0.0, 5.0]], [w0])
out = px.deposit_moments(pop, g)
n = out["n"]
expect = w0 / 8 / dV
for (i, j, k) in [(0,0,0),(1,0,0),(0,1,0),(0,0,1),(1,1,0),(0,1,1),(1,0,1),(1,1,1)]:
    assert np.isclose(n[flat(i, j, k)], expect), (i, j, k, n[flat(i, j, k)])
assert np.isclose(n.sum() * dV, w0)
assert np.allclose(out["u"][flat(0, 0, 0)], [0.0, 0.0, 5.0])
print("T1 cell-center 8-way split ok: each corner =", expect)

## T2 — Uniform plasma is exact

One particle per node with uniform weights $w = n_0 V/N_p$ must reproduce $n \equiv n_0$ and $\mathbf u \equiv \mathbf u_0$ to roundoff. This is the discrete analogue of a constant function passing through the interpolation untouched — the baseline every later phase (smoothing, Ohm) must preserve.

In [ ]:
n0, u0 = 2.5, np.array([0.5, -1.0, 2.0])
d = L / N
xs = np.array([[i*d, j*d, k*d] for i in range(N) for j in range(N) for k in range(N)])
vs = np.tile(u0, (xs.shape[0], 1))
pop = make_pop(xs, vs, np.full(xs.shape[0], n0 * L**3 / xs.shape[0]))
out = px.deposit_moments(pop, g)
assert np.allclose(out["n"], n0, rtol=1e-12), np.abs(out["n"] - n0).max()
assert np.allclose(out["u"], np.tile(u0, (N**3, 1)), rtol=1e-12)
print("T2 uniform plasma exact ok: max |n - n0| =", np.abs(out["n"] - n0).max())

## T3 — Charge conservation on a random cloud

CIC only *redistributes* weight among nodes, so $\sum_j n_j\,dV = \sum_p w_p$ up to roundoff for any particle arrangement. Here we use the object API (`Moments.setup(geom)` + `deposit(pop)`) instead of the one-call helper, and check `total()`.

In [ ]:
rng = np.random.default_rng(0)
xr = rng.uniform(0, L, size=(5000, 3))
vr = rng.normal(size=(5000, 3))
wr = rng.uniform(0.5, 1.5, size=5000)
pop = make_pop(xr, vr, wr)
mo = px.Moments()
mo.setup(g)
mo.deposit(pop)
assert np.isclose(mo.total(), wr.sum(), rtol=1e-12), (mo.total(), wr.sum())
print("T3 conservation ok: total=%.12g sum(w)=%.12g" % (mo.total(), wr.sum()))

## T4 — Periodic wrap across the boundary

The domain is periodic, so a particle at $x = L - \epsilon$ sits $\epsilon$ left of node $0$ (equivalently, right of node $N-1$). Its CIC cloud splits across the face: almost everything lands on node $0$, a sliver wraps onto node $N-1$ via `GridGeom::idx` modulo. Total weight must still be exact — nothing is lost at the boundary.

In [ ]:
eps = 1e-6 * d
pop = make_pop([[L - eps, 0.0, 0.0]], [[0.0, 0.0, 0.0]], [w0])
out = px.deposit_moments(pop, g)
n = out["n"]
assert np.isclose(n.sum() * dV, w0)  # nothing lost
assert n[flat(0, 0, 0)] > 0.99 * w0 / dV   # particle is eps left of node 0
assert n[flat(N - 1, 0, 0)] > 0.0          # wrapped sliver present
print("T4 periodic wrap ok: node0 =", n[flat(0,0,0)], ", wrapped sliver =", n[flat(N-1,0,0)])

## T5 — Bulk velocity is the density-weighted mean

Two cold beams sharing one cell with weights 1 and 3 and velocities $+10$ and $-4$: each touched node must report $(10\cdot 1 - 4\cdot 3)/4 = -0.5$. This is the first-moment half of the slide's $\mathbf v_{ij}$ formula.

In [ ]:
pop = make_pop([[c, c, c], [c, c, c]], [[10.0, 0.0, 0.0], [-4.0, 0.0, 0.0]], [1.0, 3.0])
out = px.deposit_moments(pop, g)
assert np.allclose(out["u"][flat(0, 0, 0)], [-0.5, 0.0, 0.0])
assert np.isclose(out["n"].sum() * dV, 4.0)
print("T5 two-beam mean velocity ok: u =", out["u"][flat(0, 0, 0)])

## T6 — PPC vocabulary smoke test

Phase-1 bodies in `src/ppc.cpp` are deposit-only passes (no fields/Ohm yet), but the Screenshot-2 call chain already runs end to end with final ≤3-arg signatures: `init_state(pop, geom, params)`, `predictor1(s, p)`, `predictor2(s, p1, p)`, `corrector(s, p2, p)`, `advance(s, p)`. Phase 4 will only fill in bodies.

In [ ]:
sp = px.StepParams()
sp.dt = 0.01
s = px.init_state(make_pop(xr[:1000], vr[:1000], wr[:1000]), make_geom(), sp)
p1 = px.predictor1(s, sp)
p2 = px.predictor2(s, p1, sp)
s2 = px.corrector(s, p2, sp)
s3 = px.advance(s, sp)
for st in (s, s2, s3):
    assert len(st.moments.n_array()) == N**3
print("T6 predictor1/predictor2/corrector/advance callable ok")

## T7 — Shot-noise scaling $\sigma_n/\bar n \propto 1/\sqrt{N}$

With the grid fixed ($8^3$ cells) and uniform random positions, each cell holds on average $N_{ppc} = N_p/n_{cells}$ particles. Counting statistics gives relative rms fluctuations $\sim 1/\sqrt{N_{ppc}}$ — the PIC discreteness noise that later phases (smoothing, Ohm Hall term) must keep under control. On a log-log plot of noise vs $N_p$ the slope must be $-1/2$. We also check the mean stays at $n_0 = 1$, i.e. the deposition is unbiased.

In [ ]:
ncell = 8
gN = make_geom(n=ncell)
Ns = [2000, 8000, 32000, 128000]
rel = []
rng = np.random.default_rng(1)
for Np in Ns:
    x = rng.uniform(0, L, size=(Np, 3))
    w = np.full(Np, L**3 / Np)  # n0 = 1
    out = px.deposit_moments(make_pop(x, np.zeros((Np, 3)), w), gN)
    nn = np.asarray(out["n"])
    assert abs(nn.mean() - 1.0) < 0.05, (Np, nn.mean())
    rel.append(nn.std() / nn.mean())
slope = np.polyfit(np.log(Ns), np.log(rel), 1)[0]
print("rel. rms:", [round(r, 4) for r in rel], "slope = %.3f (expect -0.5)" % slope)
assert all(a > b for a, b in zip(rel, rel[1:])), "noise must decrease with N"
assert -0.62 < slope < -0.38, slope
plt.figure()
plt.loglog(Ns, rel, "o-", label="measured")
plt.loglog(Ns, rel[0] * (np.array(Ns) / Ns[0]) ** -0.5, "--", label="1/sqrt(N) ref")
plt.xlabel("particles N"); plt.ylabel("std(n)/mean(n)")
plt.title("PIC shot noise scaling (slope %.2f)" % slope)
plt.legend(); plt.grid(True, which="both", ls=":")
plt.show()
print("T7 1/sqrt(N) scaling ok")

## Summary

All moment tests passed: exact single-particle splits (T0/T1), uniform plasma (T2), conservation (T3), periodicity (T4), velocity mean (T5), PPC vocabulary (T6), and $1/\sqrt{N}$ shot noise (T7).

**Next:** Phase 2 adds the `YeeGrid` (fields-to-particles gather, Faraday, `average_fields`) — Screenshot 1 left + bottom boxes — reusing these tested moments unchanged.